In [1]:
import numpy as np
import os
import sys
from stable_baselines3 import SAC
import os
import torch
import requests
import pandas as pd
from IPython.display import clear_output

In [2]:
sys.path.insert(0,'../boptestGym')
from boptestGymEnv import BoptestGymEnv
from boptestGymEnv import NormalizedObservationWrapper

In [3]:
url_api = 'https://api.boptest.net'
url = "http://localhost:80"

In [4]:
class BoptestGymEnvCustomReward(BoptestGymEnv):
    def get_reward(self):
        # Compute BOPTEST core kpis
        kpis = requests.get('{0}/kpi/{1}'.format(self.url, self.testid)).json()['payload']

        ener_coef = 1.0
        temp_coef = 1.0
        
        # todo: search for best reward function
        reward = - ((kpis['tdis_tot']*temp_coef) + (kpis['ener_tot']*ener_coef))

        # Record current objective integrand for next evaluation
        self.objective_integrand = reward
        return reward

In [5]:
def create_env():
    env = BoptestGymEnv(url                   = url,
                    testcase              = 'bestest_hydronic_heat_pump',
                    actions               = ['oveTSet_u'],
                    observations          = {'reaTZon_y':(273.,324.),
                                             'weaSta_reaWeaTDryBul_y':(257.,305.),
                                             'weaSta_reaWeaHDirNor_y':(0.,862.),
                                             'reaQFloHea_y':(0.,  15000.),
                                            },
                    random_start_time     = False,
                    start_time            = 305*24*3600, #nov 1 like the paper
                    max_episode_length    = 61 * 24*3600, #2 month testing period
                    warmup_period         = 24*3600,
                    predictive_period     = 0,
                    regressive_period     = 4*1800,
                    step_period           = 1800)
    env = NormalizedObservationWrapper(env)
    return env

In [16]:
for data_regime in [ 27]:
    for training_episodes in [ 25]:
        print(f"training episode: {training_episodes}")
        model= SAC.load(f"Saved Models/SAC_temp_1_{data_regime}_{training_episodes}.zip")
        env = create_env()
        done = False
        obs, _ = env.reset()
        rows = []

        i=0
        while i<=2928:
            # Clear the display output at each step
            clear_output(wait=True)
    
            # Compute control signal
            action, _ = model.predict(obs, deterministic=True)
            kpis = requests.get('{0}/kpi/{1}'.format(url, env.testid)).json()['payload']

            # Print the current operative temperature and decided action
            print('-------------------------------------------------------------------')
            print("obs: %s"%obs)
            print("act: %s"%action)
            print("%s /2928"%i)
            print('-------------------------------------------------------------------')
            i+=1
            # Implement action
            rows.append({
                "T_zone": obs[0],
                "t_out": obs[1],
                "Psol_Wm2": obs[2],
                "action": action,
                "energy_kWh": kpis["ener_tot"],
                "discomfort": kpis["tdis_tot"]
            })
            obs, reward, terminated, truncated, info = env.step(action)  # send the action to the environment
            done = (terminated or truncated)

        df = pd.DataFrame(rows)
        df.to_csv(f"Testing/SAC_temp_1_{data_regime}_{training_episodes}.csv", index=False)
        env.stop()
        del model

-------------------------------------------------------------------
obs: [-0.11489689 -0.3270836  -1.          0.2906145  -0.11665851 -0.11823708
 -0.11884385 -0.11983955 -0.336806   -0.3479169  -0.3562495  -0.3645833
 -1.         -1.         -1.         -1.         -0.15247363  0.26765525
 -0.30832714  0.25473344]
act: [294.6264]
2928 /2928
-------------------------------------------------------------------


In [7]:
env.stop()